# MCP Client/Host Security and Capability Negotiation

This notebook builds the host-side half of the Northstar Support MCP integration. It uses the official Python SDK in memory, so discovery and tool calls are real protocol operations while package verification, subprocess isolation, TLS, and OAuth remain explicit production upgrades.

The invariant is: **a server advertises; the trusted host verifies, filters, authorizes, validates, and revokes.**

## 1. Scenario and trust boundary

A reviewed ticket server exposes two read-only tools, one resource, and one prompt. An update self-reports the same name and version but adds filesystem access, widens a tenant parameter, injects instructions, and requests public caching. We will prove that successful MCP discovery does not approve that update.

In [ ]:
import runpy
from dataclasses import asdict, replace
from datetime import UTC, datetime
from pathlib import Path

module = runpy.run_path(Path('lab.py'))
NOW = datetime(2026, 9, 21, 12, 0, tzinfo=UTC)
print('Loaded reusable host policy and two MCP server fixtures.')

## 2. Discover the complete modern contract

`Client(..., mode='auto')` probes `server/discover` and reports the negotiated version. The reusable inspector then refreshes every paginated tool, resource, resource-template, and prompt list under page, item, and byte budgets. Cache hints and server instructions are part of the reviewed snapshot.

In [ ]:
trusted = await module['inspect_candidate'](module['trusted_mcp'])
{
    'protocol_version': trusted.protocol_version,
    'supported_versions': trusted.supported_versions,
    'self_reported_server': (trusted.advertised_name, trusted.advertised_version),
    'tool_names': trusted.tool_names,
    'resource_count': len(trusted.resources),
    'template_count': len(trusted.resource_templates),
    'prompt_count': len(trusted.prompts),
    'cache_hints': [asdict(hint) for hint in trusted.cache_hints],
    'snapshot_digest': trusted.digest,
}

## 3. Attack: self-reported identity plus capability drift

The malicious server uses the same display name and version. Those fields help a UI and logs, but the MCP specification does not make them verified identity. Compare complete contracts instead of trusting labels or tool names.

In [ ]:
attacker = await module['inspect_candidate'](module['drifted_mcp'])
assert attacker.advertised_name == trusted.advertised_name
assert attacker.advertised_version == trusted.advertised_version
assert attacker.digest != trusted.digest
{
    'same_self_reported_identity': True,
    'trusted_tools': trusted.tool_names,
    'attacker_tools': attacker.tool_names,
    'attacker_cache_scopes': sorted({hint.scope for hint in attacker.cache_hints}),
    'digest_match': attacker.digest == trusted.digest,
}

## 4. Separate review from admission

Discovery creates an untrusted candidate. A separate reviewer decision binds the accepted snapshot to the host-owned server ID, exact launch specification, verified artifact digest, owner, policy version, and expiry. Runtime admission rechecks those facts before opening the MCP client.

In [ ]:
installation = module['reviewed_installation']()
review = module['make_review_record'](installation, trusted, now=NOW)
registry = module['HostRegistry']()
registry.approve(review)
{
    'host_server_id': review.server_id,
    'artifact_digest': review.artifact_digest,
    'launch_spec': review.launch_spec,
    'review_expires': review.expires_at.isoformat(),
    'exposed_names': [grant.exposed_name for grant in review.grants],
}

## 5. Authorize a namespaced invocation

The authenticated analyst identity comes from application state. The host exposes stable `support.*` names, validates arguments before the server handler, and validates structured output again before returning it. Raw arguments and the subject are not written to the audit record.

In [ ]:
principal = module['AuthenticatedPrincipal'](
    subject='analyst-42', tenant_id='acme', permissions=frozenset({'ticket:read'})
)
cache = module['CapabilityCache']()
audit = []
module['reset_runtime_evidence']()
async with module['SecureHostConnection'](
    target=module['trusted_mcp'], installation=installation, principal=principal,
    registry=registry, cache=cache, audit=audit, clock=lambda: NOW,
) as host:
    visible = host.visible_tools
    result = await host.call_tool(
        'support.ticket.read', {'ticket_id': 'acme-7'}, trace_id='notebook-read'
    )
{
    'visible_tools': visible,
    'validated_result': result.model_dump(),
    'server_calls': dict(module['SERVER_CALLS']),
    'last_audit_event': asdict(audit[-1]),
}

## 6. Failure injection: authorization and drift

The analyst cannot search merely because the server advertises search. The impersonating server also cannot reach the model merely because discovery succeeds. Both failures produce stable host reason codes.

In [ ]:
async def denial_codes():
    permission_code = None
    async with module['SecureHostConnection'](
        target=module['trusted_mcp'], installation=installation, principal=principal,
        registry=registry, cache=cache, audit=audit, clock=lambda: NOW,
    ) as host:
        try:
            await host.call_tool(
                'support.ticket.search', {'query': 'payment', 'limit': 5},
                trace_id='notebook-denied-search',
            )
        except module['HostDenied'] as error:
            permission_code = error.code
    drift_code = None
    try:
        async with module['SecureHostConnection'](
            target=module['drifted_mcp'], installation=installation, principal=principal,
            registry=registry, cache=cache, audit=audit, clock=lambda: NOW,
        ):
            pass
    except module['HostDenied'] as error:
        drift_code = error.code
    return permission_code, drift_code

permission_code, drift_code = await denial_codes()
assert permission_code == 'PERMISSION_DENIED'
assert drift_code == 'CAPABILITY_DRIFT'
{'permission_denial': permission_code, 'drift_denial': drift_code}

## 7. Cache partition and revocation

A server's `ttlMs` and `cacheScope` are hints, not authority. The host caps TTL at five seconds and includes the authorization context in its cache key. Revocation must block an already-admitted connection before another handler runs and remove cached capability data.

In [ ]:
evidence = await module['run_scenario']()
assert evidence.malicious_server_denied
assert evidence.cross_permission_denied
assert evidence.revocation_denied_active_connection
assert evidence.cache_entries_after_revocation == 0
{
    'protocol_version': evidence.protocol_version,
    'server_calls': evidence.server_calls,
    'cache_entries_after_revocation': evidence.cache_entries_after_revocation,
    'decision_reasons': [event.reason_code for event in evidence.audit_events],
}

## 8. Interpret the evidence

The scenario demonstrates one valid read, zero forbidden search executions, drift denial before publication, and active-use denial after revocation. These are deterministic control checks—not a production benchmark. A real evaluation should label unchanged, drifted, malformed, unauthorized, expired, and revoked cases; report the denominator for block and false-denial rates; and measure revocation propagation separately from detection time.

## 9. Production upgrades and exercises

Replace the demo installation fact with verified Sigstore/SLSA evidence; run local servers in an OS sandbox; enforce exact HTTPS origin and SSRF-safe OAuth discovery for remote servers; persist reviews and revocations transactionally; use OPA, Cedar, or another policy engine only when centralized governance warrants it; and emit redacted OpenTelemetry/security events. Test every supported SDK and protocol revision.

Exercises:

1. Add a second server with a colliding raw tool name and prove namespace isolation.
2. Add an exact-command drift case and prove that no MCP discovery occurs.
3. Add a concurrent revocation-versus-call test and define the winning atomic transition.
4. Design a single-use approval receipt for a side-effecting tool; do not use an `approved` Boolean.

## Reflection

A server keeps its name and version but changes an annotation, an optional schema field, and `cacheScope`. Which facts require renewed review, which cache keys must be invalidated, and which checks still run on the next tool call?